**FrameNet to WordNet mapping**

# Librerie

In [ ]:
import hashlib
import string
import random
import nltk
from nltk.corpus import framenet as fn
from nltk.corpus import wordnet as wn, stopwords
from nltk.tokenize import word_tokenize

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('framenet_v17')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package framenet_v17 to /root/nltk_data...
[nltk_data]   Package framenet_v17 is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# 0. Individuazione di un set di frame

In [ ]:
def getFrameSetForStudent(full_name):

    #Scrivo il nome tutto minuscolo e senza spazi extra
    full_name = full_name.lower().strip()

    #Dal nome, applicando l'algoritmo di hash SHA-256, ottengo una stringa esadecimale, poi convertita in un intero molto grande
    name_hash = int(hashlib.sha256(full_name.encode()).hexdigest(), 16)

    #Inizializzo il generatore di numeri casuali con il seme calcolato sopra.
    #Così la selezione dei frame sarà randomica (ma sempre la stessa) per ogni studente.
    random.seed(name_hash)

    #Recupero tutti i frame disponibili in FrameNet e li converto in una lista.
    all_frames = list(fn.frames())

    #Estraggo 5 frame casuali dalla lista, in modo che siano sempre gli stessi per ciascun studente, grazie al seed impostato prima.
    sampled_frames = random.sample(all_frames, 5)

    #Creo una lista finale con ID e nome del frame
    frame_set = []
    for frame in sampled_frames:
        frame_set.append({
            'ID': frame.ID,
            'frame': frame.name
        })

    return frame_set

In [ ]:
# Caricamento dei FrameSet individuali
frames_LDS = getFrameSetForStudent("Luca Di Salvo")
frames_LG = getFrameSetForStudent("Luca Grandi")
frames_LS = getFrameSetForStudent("Lorenzo Sala")

print("Frame per Luca Di Salvo:")
for f in frames_LDS:
    print(f"ID: {f['ID']}  frame: {f['frame']}")

print("\nFrame per Luca Grandi:")
for f in frames_LG:
    print(f"ID: {f['ID']}  frame: {f['frame']}")

print("\nFrame per Lorenzo Sala:")
for f in frames_LS:
    print(f"ID: {f['ID']}  frame: {f['frame']}")

#Merge dei FrameSet
merged_frames = frames_LDS + frames_LG + frames_LS

# Stampa dell’elenco unificato e pulito
print("\n\nMerged Frames:")
for f in merged_frames:
    print(f"ID: {f['ID']}  frame: {f['frame']}")

Frame per Luca Di Salvo:
ID: 31  frame: Scrutiny
ID: 1623  frame: Respond_to_proposal
ID: 76  frame: Distributed_position
ID: 2658  frame: Suicide_attack
ID: 920  frame: Cause_fluidic_motion

Frame per Luca Grandi:
ID: 351  frame: Grinding
ID: 189  frame: Quantified_mass
ID: 232  frame: Process_end
ID: 2262  frame: Impression
ID: 1085  frame: Make_acquaintance

Frame per Lorenzo Sala:
ID: 181  frame: Activity_paused_state
ID: 80  frame: Frequency
ID: 2075  frame: Punctual_perception
ID: 1518  frame: Withdraw_from_participation
ID: 403  frame: Achieving_first


Merged Frames:
ID: 31  frame: Scrutiny
ID: 1623  frame: Respond_to_proposal
ID: 76  frame: Distributed_position
ID: 2658  frame: Suicide_attack
ID: 920  frame: Cause_fluidic_motion
ID: 351  frame: Grinding
ID: 189  frame: Quantified_mass
ID: 232  frame: Process_end
ID: 2262  frame: Impression
ID: 1085  frame: Make_acquaintance
ID: 181  frame: Activity_paused_state
ID: 80  frame: Frequency
ID: 2075  frame: Punctual_perception
ID: 

# 1. Mapping automatico con bag-of-words (BoW) e disambiguazione


In [ ]:
#PreProcessing del testo: testo in minuscolo, rimozione della punteggiatura e stopwords, divisione parole in token
def preprocess_text(text):
    if isinstance(text, list):
        text = ' '.join(text)
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]




#Costruisco, a partire dal nome di un Frame, il contesto di FrameNet (Ctx_w) da definizione + FEs (frame Elements)
#Recupero la definizione di quel Frame e dei suoi FEs, unendoli in un unico blocco di testo, andando a creare così il contesto
def get_framenet_context(frame_name):
    frame = fn.frame_by_name(frame_name)
    context = frame.definition
    for fe in frame.FE.values():
        if 'definition' in fe:
            context += ' ' + fe['definition']
    return preprocess_text(context)




#Costruisco, a partire da un synset WordNet in input, il contesto di WordNet (Ctx_s):
#estraggo Definizioni (gloss), Esempi, Iperonimi (categorie più generali) e Iponimi (sottocategorie).
#Restituisce un set di parole preprocessato che va a comporre il contesto
def build_synset_context(syn):
    tokens = preprocess_text(syn.definition())
    for ex in syn.examples():
        tokens.extend(preprocess_text(ex))
    for rel in syn.hypernyms() + syn.hyponyms():
        tokens.extend(preprocess_text(rel.definition()))
    return set(tokens)




#Calcola l'overlap (parole in comune tra Ctx_w di FrameNet e Ctx_s di WordNet)
#Confronta il contesto del termine da disambiguare (Ctx_w) e il contesto di ciascun synset candidato (Ctx_s)
#Conta le parole comuni (intersezione) tra i due contesti e restituisce lo score semantico
def compute_overlap_score(context_w, context_s):
    return len(set(context_w).intersection(context_s)) + 1




#Riceve in input un termine, il contesto semantico Ctw_x e la parte del discorso
#Disambiguazione: restituisce il synset con overlap massimo LESK
def disambiguate_term(term, context_words, pos=None):

    #Per quel termine recupera tutti i synset candidati da WordNet. Se non li ha return None
    candidates = wn.synsets(term, pos=pos)
    if not candidates:
        print(f"Nessun synset trovato per '{term}'")
        return None

    #Per ogni synset candidato costruisce il contesto Ctx_s, calcola l'overlap con Ctw_x e salva il synset col valore più alto
    best_score = -1
    best_syn = None
    for syn in candidates:
        ctx_s = build_synset_context(syn)
        score = compute_overlap_score(context_words, ctx_s)
        if score > best_score:
            best_score = score
            best_syn = syn

    print(f"Term: {term} → Synset: {best_syn.name()} – {best_syn.definition()} (score={best_score})")
    return best_syn

In [ ]:
#Funzione che mappa il nome del frame a un synset WordNet, con l'aggiustamento per le multiwords
def map_frame_name_to_synset(frame_name):
    context = get_framenet_context(frame_name)

    #Divido il nome del frame in singole parole/token qualora ci sia l'underscore
    tokens = frame_name.split("_")

    # Controllo se l'intera multi-word è presente su WordNet
    if len(tokens) > 1:
        best_syn = disambiguate_term(frame_name, context)
        if best_syn is not None:
            return best_syn.name()

    #Faccio il POS tagging delle parole per capire quali parti del discorso ottengo
    tagged = nltk.pos_tag(tokens)

    nouns = [tok for tok, tag in tagged if tag.startswith("NN")]
    verbs = [tok for tok, tag in tagged if tag.startswith("VB")]

    pos = None
    if verbs:
        key_term = verbs[0] #VERB + NOUN --> la parte più importante è il verbo, prendo il primo
        pos = wn.VERB
    elif nouns:
        key_term = nouns[0] #NOUN + ADJ --> la parte più importante è il nome (che in inglese segue sempre l'aggettivo), e prendo l'ultimo
        pos = wn.NOUN
    else:
        key_term = tokens[-1] #altrimenti disambiguo l'ultima parola
        pos = tagged[-1][1]

    best_syn = disambiguate_term(key_term, context, pos=pos)
    return best_syn.name() if best_syn else None




#Funzione che mappa i Frame Elements (FEs) ai synset di WordNet più rilevanti
def map_fes_to_synsets(frame_name):
    frame = fn.frame_by_name(frame_name)
    fe_results = {}

    #Itero su tutti i Frame Elements del frame scelto
    for fe in frame.FE.values():
        #Nome del Frame Element, es. "Agent", "Instrument"
        fe_name = fe['name']
        #Recupero la definizione (può non esserci)
        fe_def = fe.get('definition', '')

        #Se non c'è definizione, salta l'elemento
        if not fe_def:
            print(f"Nessuna definizione per FE '{fe_name}'")
            continue

        print(f"\nFE: {fe_name}")
        #Preprocessing della definizione (che è il contesto Ctx_w)
        ctx_w = preprocess_text(fe_def)

        #Disambiguazione: cerca il synset WordNet con maggiore overlap con il contesto
        best_syn = disambiguate_term(fe_name, ctx_w, pos=wn.NOUN)

        # Salva il risultato nel dizionario; se nulla è trovato, mette None
        fe_results[fe_name] = best_syn.name() if best_syn else None

    return fe_results





#Funzione che mappa i Lexical Units (LUs) di un frame FrameNet al synset più appropriato di WordNet
def map_lus_to_synsets(frame_name):

    frame = fn.frame_by_name(frame_name)
    lu_results = {}

    # Itero su tutte i Lexical Units del frame
    for lu_name, lu_data in frame.lexUnit.items():
        #Es: 'analyze.v', 'scrutiny.n'
        lemma_with_pos = lu_data["name"]
        # Gloss/definizione della Lexical Unit
        definition = lu_data.get("definition", "")

        #Estrae Lemma e Parte del Discorso dalla notazione LU (ex: look.v -> look, v)
        if "." in lemma_with_pos:
            lemma, pos_letter = lemma_with_pos.split(".")

            tagged = nltk.pos_tag(lemma.split(' '))
            nouns = [tok for tok, tag in tagged if tag.startswith("NN")]
            verbs = [tok for tok, tag in tagged if tag.startswith("VB")]

            if verbs:
                lemma = verbs[0] #VERB + NOUN --> la parte più importante è il verbo, prendo il primo
            elif nouns:
                lemma = nouns[0] #NOUN + ADJ --> la parte più importante è il nome (che in inglese segue sempre l'aggettivo), e prendo l'ultimo
            else:
                lemma = lemma.split(' ')[-1] #altrimenti disambiguo l'ultima parola

            #Mapping della lettera PoS (v, n, a, r) alla costante di WordNet
            pos_map = {"n": wn.NOUN, "v": wn.VERB, "a": wn.ADJ, "r": wn.ADV}

            pos = pos_map.get(pos_letter, None)
        else:
            lemma = lemma_with_pos
            tagged = nltk.pos_tag(lemma.split(' '))
            nouns = [tok for tok, tag in tagged if tag.startswith("NN")]
            verbs = [tok for tok, tag in tagged if tag.startswith("VB")]

            if verbs:
                lemma = verbs[0] #VERB + NOUN --> la parte più importante è il verbo, prendo il primo
            elif nouns:
                lemma = nouns[0] #NOUN + ADJ --> la parte più importante è il nome (che in inglese segue sempre l'aggettivo), e prendo l'ultimo
            else:
                lemma = lemma.split(' ')[-1] #altrimenti disambiguo l'ultima parola
            pos = None  # Se non c'è PoS, cerca in tutti i synset

        print(f"\nLU: {lemma_with_pos}")

        #Se non c'è una definizione, uso solo il lemma per creare il contesto
        if not definition:
            print(f"Nessuna definizione per LU '{lemma_with_pos}', uso solo il lemma.")
            context = [lemma]
        else:
            #Altrimenti, preprocessa la definizione per ottenere il contesto
            context = preprocess_text(definition)

        #Disambiguazione: trova il synset migliore basandosi su overlap semantico
        best_syn = disambiguate_term(lemma, context, pos=pos)

        # Salva il risultato finale: nome LU → nome synset (es. 'search.v': 'search.v.01')
        lu_results[lemma_with_pos] = best_syn.name() if best_syn else None

    return lu_results

## Print dei vari mapping (Frame Name, FEs e LUs)

In [ ]:
# MAPPING Frame Name per merged frames
for frame in merged_frames:
    frame_name = frame["frame"]
    print(f"\nMAPPING FRAME NAME: {frame_name}")
    synset = map_frame_name_to_synset(frame_name)

    print("RISULTATO:")
    if synset:
        print(f"{frame_name} → {synset}")
    else:
        print(f"Nessun synset trovato per '{frame_name}'")


MAPPING FRAME NAME: Scrutiny
Term: Scrutiny → Synset: examination.n.01 – the act of examining something closely (as for mistakes) (score=10)
RISULTATO:
Scrutiny → examination.n.01

MAPPING FRAME NAME: Respond_to_proposal
Nessun synset trovato per 'Respond_to_proposal'
Nessun synset trovato per 'Respond'
RISULTATO:
Nessun synset trovato per 'Respond_to_proposal'

MAPPING FRAME NAME: Distributed_position
Nessun synset trovato per 'Distributed_position'
Term: Distributed → Synset: circulate.v.03 – cause be distributed (score=5)
RISULTATO:
Distributed_position → circulate.v.03

MAPPING FRAME NAME: Suicide_attack
Nessun synset trovato per 'Suicide_attack'
Term: Suicide → Synset: suicide.n.01 – the act of killing yourself (score=3)
RISULTATO:
Suicide_attack → suicide.n.01

MAPPING FRAME NAME: Cause_fluidic_motion
Nessun synset trovato per 'Cause_fluidic_motion'
Term: Cause → Synset: causal_agent.n.01 – any entity that produces an effect or is responsible for events or results (score=10)
RIS

In [ ]:
# MAPPING FEs per merged frames
for frame in merged_frames:
    frame_name = frame["frame"]
    print(f"\nMAPPING FEs per frame: {frame_name}")
    fe_synsets = map_fes_to_synsets(frame_name)

    print("\nRISULTATO: ")
    for fe, syn in fe_synsets.items():
        print(f"  {fe} → {syn}")


MAPPING FEs per frame: Scrutiny

FE: Cognizer
Nessun synset trovato per 'Cognizer'

FE: Ground
Term: Ground → Synset: land.n.04 – the solid part of the earth's surface (score=2)

FE: Phenomenon
Term: Phenomenon → Synset: phenomenon.n.01 – any state or process known through the senses rather than by intuition or reasoning (score=2)

FE: Manner
Term: Manner → Synset: manner.n.01 – how something is done or how it happens (score=2)

FE: Means
Term: Means → Synset: means.n.01 – how a result is obtained or an end is achieved (score=2)

FE: Degree
Term: Degree → Synset: degree.n.01 – a position on a scale of intensity or amount or quality (score=3)

FE: Direction
Term: Direction → Synset: direction.n.01 – a line leading to a place or point (score=3)

FE: Purpose
Term: Purpose → Synset: purpose.n.01 – an anticipated outcome that is intended or that guides your planned actions (score=2)

FE: Medium
Term: Medium → Synset: medium.n.01 – a means or instrumentality for storing or communicating inf

In [ ]:
# MAPPING LUs per merged frames
for frame in merged_frames:
    frame_name = frame["frame"]
    print(f"\nMAPPING LUs per frame: {frame_name}")
    lu_synsets = map_lus_to_synsets(frame_name)

    print("\nRISULTATO:")
    for lu, syn in lu_synsets.items():
        print(f"  {lu} → {syn}")



MAPPING LUs per frame: Scrutiny

LU: analyse.v
Term: analyse → Synset: analyze.v.01 – consider in detail and subject to an analysis in order to discover essential features or meaning (score=4)

LU: analysis.n
Term: analysis → Synset: analysis.n.01 – an investigation of the component parts of a whole and their relations in making up the whole (score=5)

LU: investigate.v
Term: investigate → Synset: investigate.v.02 – conduct an inquiry or investigation of (score=2)

LU: investigation.n
Term: investigation → Synset: investigation.n.02 – the work of inquiring into something thoroughly and systematically (score=4)

LU: look.v
Term: look → Synset: look.v.01 – perceive with attention; direct one's gaze towards (score=1)

LU: perusal.n
Term: perusal → Synset: perusal.n.01 – reading carefully with intent to remember (score=1)

LU: peruse.v
Term: peruse → Synset: peruse.v.01 – examine or consider with attention and in detail (score=3)

LU: scan.v
Term: scan → Synset: scan.v.01 – examine minute

# 3. Valutazione dell'accuracy con confronto dei synset restituiti in output dal sistema con quelli annotati a mano

In [ ]:
import json

#Mapping degli elementi del primo frame non contenente multi-words
frame_name = "Scrutiny"

# Mapping automatico dei tre livelli
print(f"\nMAPPING per frame: {frame_name}")
frame_synset = map_frame_name_to_synset(frame_name)
fe_synsets = map_fes_to_synsets(frame_name)
lu_synsets = map_lus_to_synsets(frame_name)

# Costruisco la struttura dei risultati
system_scrutiny = {
    frame_name: {
        "frame_name": frame_synset,
        "FEs": fe_synsets,
        "LUs": lu_synsets
    }
}

# Salva su file JSON
with open("SystemMapping.json", "w") as f:
    json.dump(system_scrutiny, f, indent=2)


MAPPING per frame: Scrutiny
Term: Scrutiny → Synset: examination.n.01 – the act of examining something closely (as for mistakes) (score=10)

FE: Cognizer
Nessun synset trovato per 'Cognizer'

FE: Ground
Term: Ground → Synset: land.n.04 – the solid part of the earth's surface (score=2)

FE: Phenomenon
Term: Phenomenon → Synset: phenomenon.n.01 – any state or process known through the senses rather than by intuition or reasoning (score=2)

FE: Manner
Term: Manner → Synset: manner.n.01 – how something is done or how it happens (score=2)

FE: Means
Term: Means → Synset: means.n.01 – how a result is obtained or an end is achieved (score=2)

FE: Degree
Term: Degree → Synset: degree.n.01 – a position on a scale of intensity or amount or quality (score=3)

FE: Direction
Term: Direction → Synset: direction.n.01 – a line leading to a place or point (score=3)

FE: Purpose
Term: Purpose → Synset: purpose.n.01 – an anticipated outcome that is intended or that guides your planned actions (score=2)


In [ ]:
import json

with open('/content/ManualAnnotations.json') as f:
    gold = json.load(f)

with open('/content/SystemMapping.json') as f:
    system = json.load(f)

correct = 0
total = 0

#Frame key indica Scrutiny (primo senza multi-words)
for frame_key in gold:
    if frame_key not in system:
        continue  # salta se il frame non è presente nel sistema

    gold_frame = gold[frame_key]
    system_frame = system[frame_key]

    # 1. Frame name
    if gold_frame["frame_name"] == system_frame.get("frame_name"):
        correct += 1
    total += 1

    # 2. Frame Elements
    for fe, gold_syn in gold_frame["FEs"].items():
        sys_syn = system_frame["FEs"].get(fe)
        if gold_syn is None:
            continue  # salta se non è stato annotato manualmente
        total += 1
        if gold_syn == sys_syn:
            correct += 1
        else:
            print(f"FE mismatch on {frame_key} – {fe}: gold = {gold_syn}, system = {sys_syn}")


    # 3. Lexical Units
    for lu, gold_syn in gold_frame["LUs"].items():
        sys_syn = system_frame["LUs"].get(lu)
        if gold_syn is None:
            continue  # salta se non è stato annotato manualmente
        total += 1
        if gold_syn == sys_syn:
            correct += 1
        else:
            print(f"LU mismatch on {frame_key} – {lu}: gold = {gold_syn}, system = {sys_syn}")

accuracy = correct / total
print(f"\n\nAccuracy: {accuracy:.3%} ({correct}/{total} correct)")


FE mismatch on Scrutiny – Ground: gold = reason.n.01, system = land.n.04
FE mismatch on Scrutiny – Instrument: gold = instrument.n.01, system = instrumental_role.n.01
LU mismatch on Scrutiny – study.n: gold = survey.n.01, system = discipline.n.01
LU mismatch on Scrutiny – inspector.n: gold = examiner.n.02, system = inspector.n.01
LU mismatch on Scrutiny – surveyor.n: gold = surveyor.n.02, system = surveyor.n.01
LU mismatch on Scrutiny – check.v: gold = check.v.02, system = see.v.10
LU mismatch on Scrutiny – spy out the land.v: gold = spy.v.03, system = spy.v.02
LU mismatch on Scrutiny – explore.v: gold = explore.v.03, system = explore.v.02
LU mismatch on Scrutiny – pry.v: gold = intrude.v.03, system = pry.v.01


Accuracy: 84.746% (50/59 correct)
